# ⛳ ParkCaddy AR - Google Colab 3D 그린 지면 깊이 측정기

구글 코랩(Google Colab)의 **100% 보안 HTTPS 환경**에서 ParkCaddy AR 앱을 실행하고 스마트폰 카메라로 테스트하는 전용 노트북입니다.


In [ ]:
# [1단계] 필요 패키지 설치 및 HTML 앱 파일 준비
!pip install -q pyqrcode pypng
import pyqrcode
import http.server
import socketserver
import threading
import subprocess
import re
from IPython.display import HTML, display, Image

PORT = 8000
html_code = '''<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
  <title>ParkCaddy AR - 파크골프 3D 지형 측정기</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@400;700;800&family=Pretendard:wght@400;600;700&display=swap" rel="stylesheet">
  <style>
    :root {
      --fp: 'Pretendard', sans-serif; --fd: 'Outfit', sans-serif;
      --bg:#07090e; --hud:rgba(13,17,26,.92); --bdr:rgba(255,255,255,.15);
      --g:#10B981; --n:#00F0FF; --y:#F59E0B; --r:#EF4444;
      --tx:#F3F4F6; --mt:#9CA3AF;
    }
    *{box-sizing:border-box;margin:0;padding:0;user-select:none;}
    html,body{width:100%;height:100dvh;overflow:hidden;background:var(--bg);font-family:var(--fp);color:var(--tx);}
    #app{position:relative;width:100vw;height:100dvh;overflow:hidden;}

    #cam,#snap{position:absolute;top:0;left:0;width:100%;height:100%;object-fit:cover;z-index:1;}
    #snap.hidden{display:none;}
    #ar{position:absolute;top:0;left:0;width:100%;height:100%;z-index:2;}
    #lbl{position:absolute;top:0;left:0;width:100%;height:100%;z-index:3;pointer-events:none;}

    .badge{position:absolute;transform:translate(-50%,-50%);font-family:var(--fd);border-radius:8px;backdrop-filter:blur(8px);box-shadow:0 2px 8px rgba(0,0,0,.6);white-space:nowrap;display:flex;flex-direction:column;align-items:center;padding:3px 7px;line-height:1.2;}
    .bp{background:rgba(239,68,68,.88);color:#FFF;border:1px solid #FCA5A5;}
    .bn{background:rgba(0,240,255,.88);color:#07090E;border:1px solid #67E8F9;}
    .bz{background:rgba(16,185,129,.88);color:#FFF;border:1px solid #6EE7B7;}
    .bh{font-size:.74rem;font-weight:800;white-space:nowrap;}
    .bd{font-size:.62rem;font-weight:700;opacity:.95;margin-top:2px;background:rgba(0,0,0,.25);padding:1px 5px;border-radius:4px;white-space:nowrap;}

    #hud{position:absolute;top:0;left:0;width:100%;height:100dvh;z-index:10;pointer-events:none;
         display:flex;flex-direction:column;justify-content:space-between;
         padding:max(10px,env(safe-area-inset-top)) max(10px,env(safe-area-inset-right)) max(10px,env(safe-area-inset-bottom)) max(10px,env(safe-area-inset-left));}
    #hud *{pointer-events:auto;}

    /* 헤더 */
    .hdr{display:flex;flex-direction:column;gap:6px;background:var(--hud);backdrop-filter:blur(16px);border:1px solid var(--bdr);border-radius:16px;padding:8px 12px;}
    .hdr-top{display:flex;justify-content:space-between;align-items:center;gap:6px;}
    .brand{display:flex;align-items:center;gap:8px;}
    .logo{font-size:1.1rem;background:rgba(16,185,129,.2);width:32px;height:32px;border-radius:10px;display:flex;align-items:center;justify-content:center;border:1px solid var(--g);}
    .brand h1{font-family:var(--fd);font-size:1rem;font-weight:800;color:#FFF;}
    .ar{color:var(--n);}
    .hdr-r{display:flex;gap:4px;}
    .mini-ctrl{display:flex;align-items:center;gap:3px;background:rgba(0,0,0,.5);padding:3px 6px;border-radius:10px;border:1px solid var(--bdr);}
    .mc-lbl{font-size:.6rem;color:var(--mt);font-weight:700;}
    .mc-val{font-family:var(--fd);font-size:.78rem;font-weight:800;color:var(--n);min-width:44px;text-align:center;}
    .mc-btn{background:rgba(255,255,255,.1);border:none;color:#FFF;width:18px;height:18px;border-radius:4px;font-weight:800;cursor:pointer;font-size:.7rem;display:flex;align-items:center;justify-content:center;}

    /* 우측 사이드 - 슬라이더 카드만 */
    .side{position:absolute;top:105px;right:10px;display:flex;flex-direction:column;gap:8px;z-index:30;align-items:center;}
    .ico-btn{width:44px;height:44px;border-radius:12px;background:var(--hud);backdrop-filter:blur(16px);border:1.5px solid var(--bdr);color:#FFF;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:1px;cursor:pointer;}
    .ico-btn .i{font-size:1.05rem;} .ico-btn .t{font-size:.52rem;font-weight:700;color:var(--mt);}
    .sc{background:var(--hud);backdrop-filter:blur(16px);border:1.5px solid var(--bdr);border-radius:14px;padding:8px 6px;display:flex;flex-direction:column;align-items:center;gap:5px;}
    .sc-t{font-size:.55rem;font-weight:800;color:var(--n);text-align:center;line-height:1.1;}
    .sc-v{font-family:var(--fd);font-size:.8rem;font-weight:800;color:#FFF;}
    input[type=range].vs{writing-mode:vertical-lr;direction:rtl;appearance:slider-vertical;width:14px;height:90px;background:rgba(255,255,255,.2);border-radius:6px;cursor:pointer;}

    /* 히트맵 레전드 */
    .legend{position:absolute;bottom:200px;left:10px;background:var(--hud);backdrop-filter:blur(16px);border:1px solid var(--bdr);border-radius:12px;padding:6px 10px;width:155px;z-index:20;display:none;flex-direction:column;gap:4px;}
    .legend.show{display:flex;}
    .leg-h{font-size:.62rem;font-weight:700;color:var(--mt);display:flex;justify-content:space-between;align-items:center;cursor:pointer;}
    .leg-b{display:flex;flex-direction:column;gap:2px;}
    .legend.col .leg-b{display:none;}
    .leg-r{display:flex;align-items:center;justify-content:space-between;font-size:.58rem;color:#FFF;font-weight:600;}
    .leg-sw{width:14px;height:10px;border-radius:3px;border:1px solid rgba(255,255,255,.3);}

    /* 공 마커 */
    #ball{position:absolute;top:45%;left:50%;transform:translate(-50%,-50%);display:flex;flex-direction:column;align-items:center;cursor:grab;z-index:25;touch-action:none;}
    #ball.drag{cursor:grabbing;opacity:.9;}
    .b-ring{width:44px;height:44px;border:2.5px solid var(--n);border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:1.1rem;background:rgba(0,240,255,.15);backdrop-filter:blur(4px);box-shadow:0 0 15px rgba(0,240,255,.5);transition:transform .15s;}
    .b-ring.pulse{border-color:var(--y);transform:scale(1.25);background:rgba(245,158,11,.2);}
    .b-tip{margin-top:4px;font-size:.68rem;font-weight:700;background:rgba(0,0,0,.88);padding:3px 10px;border-radius:10px;color:var(--n);border:1px solid rgba(0,240,255,.5);white-space:nowrap;pointer-events:none;}

    /* 측정 진행 오버레이 */
    #meas-overlay{position:absolute;top:0;left:0;width:100%;height:100%;z-index:28;display:none;align-items:center;justify-content:center;pointer-events:none;}
    #meas-overlay.show{display:flex;}
    .meas-box{background:rgba(0,0,0,.75);border:2px solid var(--y);border-radius:20px;padding:20px 32px;text-align:center;backdrop-filter:blur(12px);}
    .meas-step{font-size:.7rem;font-weight:700;color:var(--mt);margin-bottom:4px;}
    .meas-count{font-family:var(--fd);font-size:2.8rem;font-weight:800;color:var(--y);line-height:1;}
    .meas-label{font-size:.7rem;font-weight:700;color:#E0F2FE;margin-top:4px;}

    /* 푸터 */
    .ftr{display:flex;flex-direction:column;gap:5px;background:var(--hud);backdrop-filter:blur(16px);border:1px solid var(--bdr);border-radius:18px;padding:8px 12px;margin-top:auto;z-index:15;}

    /* 측정 버튼 */
    .meas-btn{padding:12px;border:none;font-weight:800;border-radius:12px;cursor:pointer;font-size:.88rem;display:flex;align-items:center;justify-content:center;gap:6px;transition:all .2s;}
    .meas-btn:disabled{opacity:.5;cursor:not-allowed;}
    .btn-idle{background:linear-gradient(135deg,var(--g),var(--n));color:#000;box-shadow:0 4px 16px rgba(16,185,129,.35);}
    .btn-measure-2{background:linear-gradient(135deg,var(--y),#f97316);color:#000;box-shadow:0 4px 16px rgba(245,158,11,.35);}
    .btn-frozen{background:rgba(255,255,255,.1);border:1.5px solid rgba(255,255,255,.3);color:#FFF;}

    /* 상태 바 */
    .sbar{padding:5px 10px;border-radius:8px;font-size:.68rem;font-weight:700;line-height:1.35;}
    .sw{background:rgba(245,158,11,.12);border:1px solid rgba(245,158,11,.35);color:#FCD34D;}
    .so{background:rgba(16,185,129,.12);border:1px solid rgba(16,185,129,.35);color:#6EE7B7;}
    .si{background:rgba(0,240,255,.08);border:1px solid rgba(0,240,255,.25);color:#67E8F9;}
    .sf{background:rgba(59,130,246,.15);border:1px solid rgba(59,130,246,.4);color:#93C5FD;}

    /* 단계 표시 */
    .step-row{display:flex;align-items:center;gap:4px;}
    .step-dot{width:22px;height:22px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:.65rem;font-weight:800;border:1.5px solid var(--bdr);color:var(--mt);flex-shrink:0;}
    .step-dot.done{background:var(--g);border-color:var(--g);color:#000;}
    .step-dot.active{background:var(--y);border-color:var(--y);color:#000;}
    .step-line{flex:1;height:2px;background:var(--bdr);}
    .step-line.done{background:var(--g);}
    .step-txt{font-size:.58rem;color:var(--mt);font-weight:700;}

    /* 뷰 탭 */
    .vtabs{display:flex;background:rgba(0,0,0,.4);padding:3px;border-radius:10px;}
    .vb{flex:1;padding:5px;border:none;background:transparent;color:var(--mt);font-size:.68rem;font-weight:700;border-radius:6px;cursor:pointer;}
    .vb.on{background:rgba(16,185,129,.3);color:#FFF;border:1px solid var(--g);}

    /* 안내 */
    .instr{background:linear-gradient(90deg,rgba(0,240,255,.1),rgba(16,185,129,.1));border:1px solid rgba(0,240,255,.2);border-radius:8px;padding:5px 10px;font-size:.66rem;font-weight:600;color:#E0F2FE;display:flex;align-items:flex-start;gap:5px;line-height:1.4;}

    /* 카메라 권한 */
    .cb{position:absolute;top:80px;left:50%;transform:translateX(-50%);z-index:1000;width:88%;max-width:320px;pointer-events:auto;}
    .cc{background:rgba(13,17,26,.95);backdrop-filter:blur(20px);border:1.5px solid var(--n);border-radius:16px;padding:14px;text-align:center;display:flex;flex-direction:column;gap:8px;box-shadow:0 10px 30px rgba(0,0,0,.8);pointer-events:auto;}
    .btn-perm{padding:12px;background:linear-gradient(135deg,var(--g),var(--n));color:#000;font-weight:800;border:none;border-radius:10px;cursor:pointer;font-size:.9rem;pointer-events:auto;}
  </style>
</head>
<body>
<div id="app">
  <video id="cam" autoplay playsinline webkit-playsinline muted></video>
  <canvas id="snap" class="hidden"></canvas>
  <canvas id="ar"></canvas>
  <div id="lbl"></div>

  <!-- 측정 카운트다운 오버레이 -->
  <div id="meas-overlay">
    <div class="meas-box">
      <div class="meas-step" id="ov-step">1단계: 평지 기준 교정</div>
      <div class="meas-count" id="ov-count">3</div>
      <div class="meas-label">폰을 고정하세요...</div>
    </div>
  </div>

  <div id="cam-perm" class="cb">
    <div class="cc">
      <h3>📷 모바일 카메라 연동</h3>
      <p style="font-size:.72rem;color:#9CA3AF;">카메라로 지면을 비추어 3D 지형을 측정합니다.</p>
      <button id="btn-cam" class="btn-perm" onclick="window.startCameraGlobal && window.startCameraGlobal()">📷 카메라 및 센서 켜기</button>
    </div>
  </div>

  <main id="hud">
    <header class="hdr">
      <div class="hdr-top">
        <div class="brand">
          <div class="logo">⛳</div>
          <h1>PARK CADDY <span class="ar">TERRAIN 3D</span></h1>
        </div>
        <div class="hdr-r">
          <div class="mini-ctrl">
            <span class="mc-lbl">📸높이</span>
            <button class="mc-btn" id="bh-">-</button>
            <span class="mc-val" id="v-h">130cm</span>
            <button class="mc-btn" id="bh+">+</button>
          </div>
          <div class="mini-ctrl">
            <span class="mc-lbl">📐범위</span>
            <button class="mc-btn" id="br-">-</button>
            <span class="mc-val" id="v-r">±15cm</span>
            <button class="mc-btn" id="br+">+</button>
          </div>
        </div>
      </div>
    </header>

    <aside class="side">
      <button id="btn-flip" class="ico-btn"><span class="i">📷</span><span class="t">전환</span></button>
      <button id="btn-demo" class="ico-btn"><span class="i">🎬</span><span class="t">데모</span></button>
      <div class="sc">
        <div class="sc-t">🔍 굴곡<br>민감도</div>
        <div class="sc-v" id="v-s">1.0x</div>
        <input type="range" id="sl-s" class="vs" min="1" max="5" step="0.5" value="1">
      </div>
    </aside>

    <section class="legend" id="legend">
      <div class="leg-h" id="leg-h"><span>🎨 높이 기준</span><span id="leg-tog" style="font-size:.6rem;color:var(--n)">접기 ▲</span></div>
      <div class="leg-b">
        <div class="leg-r"><span>+15cm 이상 (높음)</span><div class="leg-sw" style="background:#EF4444"></div></div>
        <div class="leg-r"><span>+5 ~ +15cm</span><div class="leg-sw" style="background:#F59E0B"></div></div>
        <div class="leg-r"><span>-5 ~ +5cm (평지)</span><div class="leg-sw" style="background:#10B981"></div></div>
        <div class="leg-r"><span>-15 ~ -5cm</span><div class="leg-sw" style="background:#00F0FF"></div></div>
        <div class="leg-r"><span>-15cm 이하 (낮음)</span><div class="leg-sw" style="background:#1E40AF"></div></div>
      </div>
    </section>

    <div id="ball">
      <div class="b-ring" id="b-ring">⚪</div>
      <span class="b-tip" id="b-tip">⚪ 공 위치 (0.0 cm)</span>
    </div>

    <footer class="ftr">
      <!-- 단계 표시 -->
      <div class="step-row" id="step-row">
        <div class="step-dot active" id="s1">1</div>
        <div class="step-line" id="sl1"></div>
        <div class="step-dot" id="s2">2</div>
        <div class="step-line" id="sl2"></div>
        <div class="step-dot" id="s3">✅</div>
        <span class="step-txt" id="step-label" style="margin-left:6px">1단계: 평지에서 기준 측정</span>
      </div>

      <!-- 메인 측정 버튼 -->
      <button id="btn-meas" class="meas-btn btn-idle">🎯 평지 기준 측정 (3초)</button>

      <!-- 상태 표시 -->
      <div class="sbar sw" id="status-bar">⚠️ 평평한 지면에서 [기준 측정]을 먼저 하세요</div>

      <!-- 뷰 탭 -->
      <div class="vtabs">
        <button class="vb on" data-v="grid">3D 실측 격자</button>
        <button class="vb" data-v="heatmap">깊이 히트맵</button>
        <button class="vb" data-v="contour">등고선 뷰</button>
      </div>

      <!-- 안내 -->
      <div class="instr"><span>💡</span><span id="txt-i">평평한 지면에서 카메라를 지면으로 향하고 [기준 측정]을 누르세요.</span></div>
    </footer>
  </main>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
// ── 상수 ──────────────────────────────────────────────────────────────────
const EXAG = 6.0, ALPHA = 0.15, DN = 20;
const RANGES = [5,10,15,30,50]; // cm 범위 옵션

// ── DOM ───────────────────────────────────────────────────────────────────
const camEl   = document.getElementById('cam');
const snapEl  = document.getElementById('snap');
const arEl    = document.getElementById('ar');
const lblEl   = document.getElementById('lbl');
const permEl  = document.getElementById('cam-perm');
const instrEl = document.getElementById('txt-i');
const ballEl  = document.getElementById('ball');
const bRing   = document.getElementById('b-ring');
const bTip    = document.getElementById('b-tip');
const legend  = document.getElementById('legend');
const overlay = document.getElementById('meas-overlay');
const btnMeas = document.getElementById('btn-meas');
const statusBar = document.getElementById('status-bar');

// ── 상태 ──────────────────────────────────────────────────────────────────
let measState = 'step1_idle';

let isSim = true, stream = null;
let viewMode = 'grid';
let sensitivity = 1.0, eyeH = 130, rangeIdx = 2;
let isFrozen = false, facingMode = 'environment';

// 공 마커 (3D 지면 고정 좌표 및 정규화 좌표)
let ball3X = 0, ball3Z = -3.0;
let isDrag = false;

// 자이로
let rawP = 45, rawR = 0, fP = 45, fR = 0;

// 기준 (1단계 측정값)
let calibP = null, calibR = null;
let measuredSlopeP = 0, measuredSlopeR = 0;

// 높이 배열
let heightsM = Array.from({length:DN+1},()=>new Float32Array(DN+1));

// ── THREE.js ──────────────────────────────────────────────────────────────
const scene = new THREE.Scene();
scene.fog = new THREE.Fog(0x000000, 4.5, 9.5);
const cam3 = new THREE.PerspectiveCamera(55, innerWidth/innerHeight, 0.1, 100);
const renderer = new THREE.WebGLRenderer({canvas:arEl, alpha:true, antialias:true});
renderer.setPixelRatio(Math.min(devicePixelRatio,2));
renderer.setSize(innerWidth, innerHeight);
scene.add(new THREE.AmbientLight(0xffffff,0.9));
const dL = new THREE.DirectionalLight(0x00f0ff,1.2);
dL.position.set(5,15,10); scene.add(dL);

const cMat = new THREE.ShaderMaterial({
  uniforms:{uS:{value:1.0}},
  vertexShader:`varying float vH; void main(){vH=position.y;gl_Position=projectionMatrix*modelViewMatrix*vec4(position,1.);}`,
  fragmentShader:`
    varying float vH; uniform float uS;
    void main(){
      float h=vH/max(uS,0.01);
      float ln=abs(fract(h*30.-.5)-.5)/fwidth(h*30.);
      float a=1.-min(ln,1.);
      vec3 c=vec3(.06,.85,.45);
      if(h>.03)c=mix(vec3(.96,.62,.04),vec3(.94,.27,.27),clamp(h*4.,0.,1.));
      else if(h<-.03)c=mix(vec3(0.,.94,1.),vec3(.12,.25,.69),clamp(-h*4.,0.,1.));
      gl_FragColor=vec4(mix(c,vec3(1.),a*.85),.88);
    }
  `,
  transparent:true, side:THREE.DoubleSide, fog:false
});

let geo, gM, nP, hM, cM;
let ball3DMesh, ring3DMesh;
let distRingsGroup = new THREE.Group();
scene.add(distRingsGroup);

function updateDistanceRings() {
  while(distRingsGroup.children.length > 0) {
    distRingsGroup.remove(distRingsGroup.children[0]);
  }
  const ringDists = [1, 2, 3, 4, 5];
  for (const r of ringDists) {
    const ringGeo = new THREE.RingGeometry(r - 0.025, r + 0.025, 48);
    ringGeo.rotateX(-Math.PI / 2);
    const ringMat = new THREE.MeshBasicMaterial({
      color: 0x00f0ff,
      side: THREE.DoubleSide,
      transparent: true,
      opacity: 0.32
    });
    const ringMesh = new THREE.Mesh(ringGeo, ringMat);
    ringMesh.position.set(ball3X, 0.015, ball3Z);
    distRingsGroup.add(ringMesh);
  }
}

function buildMesh(){
  [gM,nP,hM,cM,ball3DMesh,ring3DMesh].forEach(m=>m&&scene.remove(m));
  geo = new THREE.PlaneGeometry(6.0, 8.0, DN, DN);
  geo.rotateX(-Math.PI/2); geo.translate(0,0,-3.0);
  gM = new THREE.Mesh(geo, new THREE.MeshBasicMaterial({color:0x00f0ff,wireframe:true,transparent:true,opacity:.75,fog:true}));
  scene.add(gM);
  nP = new THREE.Points(geo, new THREE.PointsMaterial({color:0x10b981,size:.12,transparent:true,opacity:.9}));
  scene.add(nP);
  geo.setAttribute('color', new THREE.BufferAttribute(new Float32Array((DN+1)*(DN+1)*3),3));
  hM = new THREE.Mesh(geo, new THREE.MeshStandardMaterial({vertexColors:true,transparent:true,opacity:.75,side:THREE.DoubleSide,fog:true}));
  hM.visible=(viewMode==='heatmap'); scene.add(hM);
  cM = new THREE.Mesh(geo, cMat);
  cM.visible=(viewMode==='contour'); scene.add(cM);

  // 3D 골프공 모델
  const bGeo = new THREE.SphereGeometry(0.12, 16, 16);
  const bMat = new THREE.MeshStandardMaterial({color:0xffffff, roughness:0.25, metalness:0.05});
  ball3DMesh = new THREE.Mesh(bGeo, bMat);
  ball3DMesh.position.set(ball3X, 0.12, ball3Z);
  scene.add(ball3DMesh);

  // 3D 지면 타겟 링
  const rGeo = new THREE.RingGeometry(0.18, 0.28, 32);
  rGeo.rotateX(-Math.PI/2);
  const rMat = new THREE.MeshBasicMaterial({color:0x00f0ff, side:THREE.DoubleSide, transparent:true, opacity:0.85});
  ring3DMesh = new THREE.Mesh(rGeo, rMat);
  ring3DMesh.position.set(ball3X, 0.02, ball3Z);
  scene.add(ring3DMesh);

  updateDistanceRings();
  computeHeights();
}

// ── 공 마커 3D 지면 레이캐스팅 & 드래그 ──────────────────────────────────
const raycaster = new THREE.Raycaster();
const groundPlane = new THREE.Plane(new THREE.Vector3(0, 1, 0), 0);
const rayHit = new THREE.Vector3();

function setBallFromScreen(cx, cy) {
  const mouse = new THREE.Vector2((cx / innerWidth) * 2 - 1, -(cy / innerHeight) * 2 + 1);
  raycaster.setFromCamera(mouse, cam3);
  if (raycaster.ray.intersectPlane(groundPlane, rayHit)) {
    ball3X = Math.max(-2.7, Math.min(2.7, rayHit.x));
    ball3Z = Math.max(-6.5, Math.min(0.5, rayHit.z));
    if (ball3DMesh) ball3DMesh.position.set(ball3X, 0.12, ball3Z);
    if (ring3DMesh) ring3DMesh.position.set(ball3X, 0.02, ball3Z);
    updateDistanceRings();
    computeHeights();
  }
}

ballEl.addEventListener('pointerdown', e => {
  isDrag = true;
  ballEl.classList.add('drag');
});

window.addEventListener('pointermove', e => {
  if (!isDrag) return;
  const cx = e.touches ? e.touches[0].clientX : e.clientX;
  const cy = e.touches ? e.touches[0].clientY : e.clientY;
  setBallFromScreen(cx, cy);
});

window.addEventListener('pointerup', () => {
  if (isDrag) {
    isDrag = false;
    ballEl.classList.remove('drag');
  }
});

// 3D 공 위치를 2D 화면 마커 좌표로 투영 (지면 고정)
function updateBallMarkerScreenPos() {
  const v = new THREE.Vector3(ball3X, 0.08, ball3Z);
  v.project(cam3);
  if (v.z < 1) {
    const sx = v.x * (innerWidth / 2) + (innerWidth / 2);
    const sy = -v.y * (innerHeight / 2) + (innerHeight / 2);
    ballEl.style.left = `${sx}px`;
    ballEl.style.top = `${sy}px`;
    ballEl.style.display = (sx >= -60 && sx <= innerWidth + 60 && sy >= -60 && sy <= innerHeight + 60) ? 'flex' : 'none';
  }
}

// ── 높이 계산 (3D 실제 미터 좌표계 기준) ──────────────────────────────────
function computeHeights(){
  if(!geo) return;
  const cols = DN + 1, rows = DN + 1;
  const pos = geo.attributes.position;
  // 기준점(공 위치) 높이: 공 위치에서의 지면 높이 산출
  const bH = measuredSlopeP * ball3Z + measuredSlopeR * ball3X;

  for(let r = 0; r < rows; r++){
    for(let c = 0; c < cols; c++){
      const idx = r * cols + c;
      const gx = pos.getX(idx);
      const gz = pos.getZ(idx);
      // 공 기준 상대 높이 (미터 단위, 평지면 0.00m)
      heightsM[r][c] = (measuredSlopeP * gz + measuredSlopeR * gx) - bH;
    }
  }
  applyMesh();
}

function applyMesh(){
  if(!geo) return;
  const pos=geo.attributes.position, colors=geo.attributes.color.array;
  const maxM=RANGES[rangeIdx]/100;
  for(let r=0;r<=DN;r++){
    for(let c=0;c<=DN;c++){
      const idx=r*(DN+1)+c;
      const hm=heightsM[r][c];
      // 과도한 왜곡(EXAG=6.0)을 없애고 직관적인 현실 스케일로 적용
      pos.setY(idx, Math.max(-1.0, Math.min(1.0, hm * sensitivity * 1.5)));
      const hcl=Math.max(-maxM,Math.min(maxM,hm));
      setColor(colors,idx,hcl*100);
    }
  }
  pos.needsUpdate=true; geo.attributes.color.needsUpdate=true;
  geo.computeVertexNormals();
  cMat.uniforms.uS.value=sensitivity;
}

function setColor(a,i,hCm){
  let r,g,b;
  if(hCm>15){r=.94;g=.27;b=.27;}
  else if(hCm>5){r=.96;g=.62;b=.04;}
  else if(hCm>=-5){r=.06;g=.85;b=.45;}
  else if(hCm>-15){r=0;g=.94;b=1;}
  else{r=.12;g=.25;b=.69;}
  a[i*3]=r;a[i*3+1]=g;a[i*3+2]=b;
}

// ── 화면 고정 / 해제 ──────────────────────────────────────────────────────
function freezeFrame(){
  if(!isSim && camEl.readyState>=2){
    snapEl.width=innerWidth; snapEl.height=innerHeight;
    const ctx=snapEl.getContext('2d');
    const vw=camEl.videoWidth||innerWidth, vh=camEl.videoHeight||innerHeight;
    const s=Math.max(innerWidth/vw, innerHeight/vh);
    ctx.drawImage(camEl,(innerWidth-vw*s)/2,(innerHeight-vh*s)/2,vw*s,vh*s);
    snapEl.classList.remove('hidden');
  }
  isFrozen=true;
}

function unfreezeFrame(){
  if(!isSim){ snapEl.classList.add('hidden'); }
  isFrozen=false;
}

// ── 단계 UI 업데이트 ──────────────────────────────────────────────────────
const RANGE_LABELS=['±5cm','±10cm','±15cm','±30cm','±50cm'];

function updateUI(){
  const s1=document.getElementById('s1'), s2=document.getElementById('s2'), s3=document.getElementById('s3');
  const sl1=document.getElementById('sl1'), sl2=document.getElementById('sl2');
  const stepLbl=document.getElementById('step-label');

  switch(measState){
    case 'step1_idle':
      s1.className='step-dot active'; s2.className='step-dot'; s3.className='step-dot';
      sl1.className='step-line'; sl2.className='step-line';
      stepLbl.textContent='1단계: 평지에서 기준 측정';
      btnMeas.className='meas-btn btn-idle';
      btnMeas.textContent='🎯 평지 기준 측정 (3초)';
      btnMeas.disabled=false;
      statusBar.className='sbar sw';
      statusBar.textContent='⚠️ 평평한 지면에서 카메라를 지면으로 향하고 측정하세요';
      instrEl.textContent='📌 1단계: 평평한 지면에 서서 카메라를 지면 방향으로 향하고 [기준 측정]을 누르세요. 3초간 폰을 고정하세요.';
      break;

    case 'step1_meas':
      btnMeas.className='meas-btn btn-idle'; btnMeas.disabled=true;
      statusBar.className='sbar si'; statusBar.textContent='⏳ 기준 교정 중... 폰을 고정하세요';
      instrEl.textContent='⏳ 폰을 움직이지 마세요! 3초간 평지 기준을 측정합니다.';
      break;

    case 'step2_idle':
      s1.className='step-dot done'; s2.className='step-dot active'; s3.className='step-dot';
      sl1.className='step-line done'; sl2.className='step-line';
      stepLbl.textContent='2단계: 측정 위치로 이동 후 측정';
      btnMeas.className='meas-btn btn-measure-2';
      btnMeas.textContent='📸 이 위치 측정 & 고정 (3초)';
      btnMeas.disabled=false;
      statusBar.className='sbar so';
      statusBar.textContent='✅ 기준 완료 — 측정할 위치로 이동 후 [측정 & 고정]을 누르세요';
      instrEl.textContent='📌 2단계: 경사를 측정할 위치(예: 공 위치)로 이동하고 [측정 & 고정]을 누르세요.';
      break;

    case 'step2_meas':
      btnMeas.className='meas-btn btn-measure-2'; btnMeas.disabled=true;
      statusBar.className='sbar si'; statusBar.textContent='⏳ 지형 측정 중... 폰을 고정하세요';
      instrEl.textContent='⏳ 폰을 움직이지 마세요! 3초간 현재 위치 지형을 측정합니다.';
      break;

    case 'frozen':
      s1.className='step-dot done'; s2.className='step-dot done'; s3.className='step-dot done';
      sl1.className='step-line done'; sl2.className='step-line done';
      stepLbl.textContent='✅ 측정 완료 — 화면 고정됨';
      btnMeas.className='meas-btn btn-frozen';
      btnMeas.textContent='🔄 다시 측정 (새 위치)';
      btnMeas.disabled=false;
      statusBar.className='sbar sf';
      statusBar.textContent='📸 지형 고정됨 — 뷰 탭으로 다양한 시각화를 확인하세요';
      instrEl.textContent='✅ 측정 완료! 상단 탭으로 3D격자/히트맵/등고선 뷰로 지형을 분석하세요. [다시 측정]으로 다른 위치를 측정할 수 있습니다.';
      break;
  }
}

// ── 측정 실행 ─────────────────────────────────────────────────────────────
function doMeasure(step, onDone){
  const ovStep=document.getElementById('ov-step');
  const ovCount=document.getElementById('ov-count');
  ovStep.textContent = step===1 ? '1단계: 평지 기준 교정' : '2단계: 현재 위치 측정';
  overlay.classList.add('show');
  bRing.classList.add('pulse');

  let samples=[], rem=3;
  ovCount.textContent=rem;

  const timer=setInterval(()=>{
    samples.push({p:fP,r:fR}); rem--;
    ovCount.textContent=Math.max(0,rem);
    if(rem<=0){
      clearInterval(timer);
      overlay.classList.remove('show');
      bRing.classList.remove('pulse');
      const ap=samples.reduce((s,x)=>s+x.p,0)/samples.length;
      const ar=samples.reduce((s,x)=>s+x.r,0)/samples.length;
      onDone(ap,ar);
    }
  },1000);
}

// ── 버튼 클릭 처리 ────────────────────────────────────────────────────────
btnMeas.addEventListener('click',()=>{
  switch(measState){
    case 'step1_idle':
      measState='step1_meas'; updateUI();
      doMeasure(1,(ap,ar)=>{
        calibP=ap; calibR=ar;
        measState='step2_idle'; updateUI();
      });
      break;

    case 'step2_idle':
      measState='step2_meas'; updateUI();
      doMeasure(2,(ap,ar)=>{
        // 1단계 기준(calibP, calibR)과의 차이로 실제 지형 경사 산출
        const dp = (calibP !== null) ? (ap - calibP) : 0;
        const dr = (calibR !== null) ? (ar - calibR) : 0;
        // 미세한 손떨림 및 0.8도 미만 오차는 평지(0)로 간주
        const effDP = Math.abs(dp) < 0.8 ? 0 : dp;
        const effDR = Math.abs(dr) < 0.8 ? 0 : dr;
        measuredSlopeP = Math.tan(effDP * Math.PI / 180);
        measuredSlopeR = Math.tan(effDR * Math.PI / 180);
        computeHeights();
        freezeFrame();
        measState='frozen'; updateUI();
      });
      break;

    case 'frozen':
      unfreezeFrame();
      measuredSlopeP = 0;
      measuredSlopeR = 0;
      computeHeights();
      measState='step2_idle'; updateUI();
      break;
  }
});

// ── 자이로 ────────────────────────────────────────────────────────────────
if(window.DeviceOrientationEvent){
  window.addEventListener('deviceorientation',e=>{
    if(e.beta!==null&&e.gamma!==null&&!isFrozen){fP=fP+ALPHA*(e.beta-fP);fR=fR+ALPHA*(e.gamma-fR);rawP=e.beta;rawR=e.gamma;}
  });
}

// ── 카메라 3D 업데이트 ────────────────────────────────────────────────────
function updateCam3(){
  // pitch: 0도=정면, 90도=수직 아래. 기본 폰 각도(beta~45~60도)에 따라 회전
  const pitchRad = THREE.MathUtils.degToRad(fP || 50);
  const rollRad = THREE.MathUtils.degToRad(fR || 0);

  // 카메라를 사용자의 실제 눈높이/손높이 위치(0, eyeH/100, 0)에 고정
  const camY = Math.max(0.6, eyeH / 100);
  cam3.position.set(0, camY, 0);

  // FPS 방식 회전: 고개를 들거나 숙일 때 화면의 3D 지면 그리드가 실제 바닥과 정확히 일치
  cam3.rotation.order = 'YXZ';
  cam3.rotation.x = -pitchRad;
  cam3.rotation.z = -rollRad;
  cam3.rotation.y = 0;
}

// ── 라벨 (공 기준 실제 물리적 거리 및 높낮이 표시) ─────────────────────────────
function updateLabels(){
  lblEl.innerHTML='';
  if(viewMode!=='grid'||!geo) return;
  const pos=geo.attributes.position, wh=innerWidth/2, hh=innerHeight/2;
  const v=new THREE.Vector3();
  const step=Math.max(2,Math.floor(DN/5));
  for(let r=0;r<=DN;r+=step){
    for(let c=0;c<=DN;c+=step){
      const idx=r*(DN+1)+c;
      const gx=pos.getX(idx);
      const gz=pos.getZ(idx);
      // 공 기준 순수 물리적 수평 거리 (항상 양수: 1.5m, 3.2m 등)
      const distM=Math.hypot(gx-ball3X, gz-ball3Z);
      
      // 공 마커 바로 위와 겹치지 않도록 필터링
      if(distM < 0.35) continue;

      v.set(gx, pos.getY(idx), gz);
      const hCm=Math.round(heightsM[r][c]*100);
      v.project(cam3);
      if(v.z<1){
        const sx=v.x*wh+wh, sy=-v.y*hh+hh;
        if(sx>35&&sx<innerWidth-35&&sy>80&&sy<innerHeight-150){
          const d=document.createElement('div');
          d.className=`badge ${hCm>1?'bp':hCm<-1?'bn':'bz'}`;
          d.style.left=`${sx}px`; d.style.top=`${sy}px`;
          d.innerHTML=`<div class="bh">높낮이: ${hCm>0?'+':''}${hCm}cm</div><div class="bd">거리: ${distM.toFixed(1)}m</div>`;
          lblEl.appendChild(d);
        }
      }
    }
  }
  bTip.textContent='⚪ 기준점 (0cm · 0.0m)';
}

// ── 데모 ──────────────────────────────────────────────────────────────────
let simT=0;
function startSim(){
  isSim=true;
  snapEl.classList.remove('hidden');
  rawP=47.2; rawR=0.8;
  const ctx=snapEl.getContext('2d');
  function draw(){
    if(!isSim) return;
    simT+=0.02;
    if(!isFrozen){
      fP=fP+ALPHA*((47.2+Math.sin(simT*.5)*.4)-fP);
      fR=fR+ALPHA*((0.8+Math.cos(simT*.7)*.3)-fR);
    }
    const w=snapEl.width=innerWidth, h=snapEl.height=innerHeight;
    ctx.fillStyle='#1E293B'; ctx.fillRect(0,0,w,h*.35);
    ctx.fillStyle='#065F46'; ctx.fillRect(0,h*.35,w,h*.65);
    requestAnimationFrame(draw);
  }
  draw();
}

// ── 카메라 ────────────────────────────────────────────────────────────────
window.startCameraGlobal = async function(){
  const btn = document.getElementById('btn-cam');
  if(btn){
    btn.disabled = true;
    btn.textContent = '⏳ 카메라 요청 중...';
  }

  // 1. iOS 모션 센서 권한
  if(typeof DeviceOrientationEvent !== 'undefined' && typeof DeviceOrientationEvent.requestPermission === 'function'){
    try{
      await DeviceOrientationEvent.requestPermission();
    }catch(e){
      console.warn('센서 권한:', e);
    }
  }

  // 2. getUserMedia 지원 확인
  if(!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia){
    alert('이 브라우저는 카메라(getUserMedia)를 지원하지 않거나 HTTPS 보안 연결이 아닙니다.');
    if(btn){ btn.disabled = false; btn.textContent = '📷 카메라 및 센서 켜기'; }
    permEl.style.display = 'none';
    startSim();
    return;
  }

  // 3. 카메라 스트림 획득
  const options = [
    { video: { facingMode: { ideal: facingMode } } },
    { video: { facingMode: facingMode } },
    { video: { facingMode: 'environment' } },
    { video: true }
  ];

  let s = null, lastErr = null;
  for(const opt of options){
    try {
      if(stream) stream.getTracks().forEach(t => t.stop());
      s = await navigator.mediaDevices.getUserMedia(opt);
      if(s) break;
    } catch(e) {
      lastErr = e;
      console.warn('카메라 시도:', e);
    }
  }

  if(!s){
    let msg = '카메라 연결에 실패했습니다.';
    if(lastErr){
      if(lastErr.name === 'NotAllowedError' || lastErr.name === 'PermissionDeniedError'){
        msg = '카메라 권한이 거부되어 있습니다.\n브라우저 주소창 왼쪽의 자물쇠/설정 아이콘을 눌러 카메라 권한을 [허용]으로 변경 후 새로고침해 주세요.';
      } else {
        msg = '카메라 오류: ' + (lastErr.message || lastErr.name);
      }
    }
    alert(msg);
    if(btn){ btn.disabled = false; btn.textContent = '📷 카메라 및 센서 켜기'; }
    permEl.style.display = 'none';
    startSim();
    return;
  }

  stream = s;
  isSim = false;
  snapEl.classList.add('hidden');
  permEl.style.display = 'none';
  camEl.muted = true;
  camEl.playsInline = true;
  camEl.srcObject = stream;
  try {
    await camEl.play();
  } catch(e) {
    console.warn('play retry:', e);
    setTimeout(() => camEl.play().catch(() => {}), 150);
  }
};

const startCamera = window.startCameraGlobal;

// ── 이벤트 ────────────────────────────────────────────────────────────────
document.getElementById('btn-cam').onclick=startCamera;
document.getElementById('btn-demo').onclick=()=>isSim?startCamera():startSim();
document.getElementById('btn-flip').onclick=()=>{facingMode=facingMode==='environment'?'user':'environment';startCamera();};

// 뷰 탭
document.querySelectorAll('.vb').forEach(b=>b.addEventListener('click',e=>{
  document.querySelectorAll('.vb').forEach(x=>x.classList.remove('on'));
  e.currentTarget.classList.add('on'); viewMode=e.currentTarget.dataset.v;
  if(gM) gM.visible=!!nP&&(nP.visible=(viewMode==='grid'));
  if(hM) hM.visible=(viewMode==='heatmap');
  if(cM) cM.visible=(viewMode==='contour');
  legend.classList.toggle('show',viewMode==='heatmap');
}));

// 레전드 접기
document.getElementById('leg-h').onclick=()=>{
  legend.classList.toggle('col');
  document.getElementById('leg-tog').textContent=legend.classList.contains('col')?'펼치기 ▼':'접기 ▲';
};

// 민감도
document.getElementById('sl-s').oninput=e=>{
  sensitivity=parseFloat(e.target.value);
  document.getElementById('v-s').textContent=`${sensitivity.toFixed(1)}x`;
  applyMesh();
};

// 촬영 높이
document.getElementById('bh-').onclick=()=>{eyeH=Math.max(80,eyeH-10);document.getElementById('v-h').textContent=`${eyeH}cm`;};
document.getElementById('bh+').onclick=()=>{eyeH=Math.min(200,eyeH+10);document.getElementById('v-h').textContent=`${eyeH}cm`;};

// 범위
document.getElementById('br-').onclick=()=>{rangeIdx=Math.max(0,rangeIdx-1);document.getElementById('v-r').textContent=RANGE_LABELS[rangeIdx];applyMesh();};
document.getElementById('br+').onclick=()=>{rangeIdx=Math.min(4,rangeIdx+1);document.getElementById('v-r').textContent=RANGE_LABELS[rangeIdx];applyMesh();};

// ── 초기화 & 루프 ─────────────────────────────────────────────────────────
buildMesh(); updateUI(); startSim();

function loop(){
  fP+=ALPHA*(rawP-fP); fR+=ALPHA*(rawR-fR);
  updateCam3();
  updateBallMarkerScreenPos();
  updateLabels();
  renderer.render(scene,cam3);
  requestAnimationFrame(loop);
}
loop();

window.addEventListener('resize',()=>{
  cam3.aspect=innerWidth/innerHeight; cam3.updateProjectionMatrix();
  renderer.setSize(innerWidth,innerHeight);
});
</script>
</body>
</html>
'''
with open('index.html', 'w', encoding='utf-8') as f:
    f.write(html_code)
print('✅ ParkCaddy AR 코랩 소스가 준비되었습니다!')


In [ ]:
# [2단계] Google Colab 화면 내부에서 즉시 테스트
display(HTML(html_code))


In [ ]:
# [3단계] 스마트폰 모바일 카메라 접속용 HTTPS 주소 & QR 코드 생성
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

def start_server():
    handler = http.server.SimpleHTTPRequestHandler
    with socketserver.TCPServer(('', PORT), handler) as httpd:
        httpd.serve_forever()

t = threading.Thread(target=start_server, daemon=True)
t.start()

proc = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', f'http://localhost:{PORT}'], stderr=subprocess.PIPE, text=True)
https_url = None
for line in proc.stderr:
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            https_url = match.group(0)
            break

if https_url:
    print('\n' + '='*60)
    print(f'🎉 스마트폰 전용 HTTPS 주소: {https_url}')
    print('='*60 + '\n')
    qr = pyqrcode.create(https_url)
    qr.png('colab_qr.png', scale=6)
    display(Image('colab_qr.png'))
else:
    print('터널 생성 중... 2단계 셀에서 바로 테스트가 가능합니다.')
